# Technical-indicator feature ablation

This notebook compares the existing base predictors with the same predictors plus MACD, RSI, Bollinger bandwidth and session VWAP distance. Both feature sets use identical eligible rows, five chronological folds, model settings and out-of-sample timestamps. The target is the next bar open-to-close return. The saved tables are loaded here; model training ran in an isolated process. Indicators use current and earlier bars only. The session VWAP uses the available IEX volume in these files.

This is an analysis notebook, not dissertation prose. A direction-persistence benchmark is included alongside the majority-class benchmark. The results are exploratory and do not establish that the indicators improve future trading performance.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
results_dir = project_root / 'results'
fold_metrics = pd.read_csv(results_dir / 'indicator_ablation_fold_metrics.csv')
predictions = pd.read_csv(results_dir / 'indicator_ablation_predictions.csv', parse_dates=['datetime', 'signal_datetime', 'next_datetime', 'entry_datetime', 'exit_datetime'])
summary = pd.read_csv(results_dir / 'indicator_ablation_summary.csv')
deltas = pd.read_csv(results_dir / 'indicator_ablation_deltas.csv')
backtest = pd.read_csv(results_dir / 'indicator_ablation_backtest_summary.csv')

## Mean fold metrics

In [2]:
display(summary.sort_values(['ETF', 'model', 'feature_set']).round(4))
display(deltas.sort_values(['ETF', 'model']).round(4))

,ETF,feature_set,model,accuracy,balanced_accuracy,f1_up,roc_auc
0,QQQ,+ technical indicators,Direction persistence baseline,0.4811,0.4811,0.4818,NaN
5,QQQ,Base features,Direction persistence baseline,0.4811,0.4811,0.4818,NaN
1,QQQ,+ technical indicators,LSTM,0.5082,0.5067,0.6041,0.5162
6,QQQ,Base features,LSTM,0.5091,0.5066,0.6335,0.5199
2,QQQ,+ technical indicators,Logistic regression,0.5021,0.5006,0.6020,0.5119
7,QQQ,Base features,Logistic regression,0.5048,0.5030,0.6188,0.5158
3,QQQ,+ technical indicators,Majority baseline,0.5037,0.5000,0.6700,NaN
8,QQQ,Base features,Majority baseline,0.5037,0.5000,0.6700,NaN
4,QQQ,+ technical indicators,XGBoost,0.5130,0.5120,0.5721,0.5183
9,QQQ,Base features,XGBoost,0.5154,0.5145,0.5744,0.5245


,ETF,comparison,model,accuracy_indicators,balanced_accuracy_indicators,f1_up_indicators,roc_auc_indicators,accuracy_base,balanced_accuracy_base,f1_up_base,roc_auc_base,accuracy_delta,balanced_accuracy_delta,f1_up_delta,roc_auc_delta
0,QQQ,+ technical indicators minus base,Direction persistence baseline,0.4811,0.4811,0.4818,NaN,0.4811,0.4811,0.4818,NaN,0.0000,0.0000,0.0000,NaN
1,QQQ,+ technical indicators minus base,LSTM,0.5082,0.5067,0.6041,0.5162,0.5091,0.5066,0.6335,0.5199,-0.0009,0.0001,-0.0295,-0.0037
2,QQQ,+ technical indicators minus base,Logistic regression,0.5021,0.5006,0.6020,0.5119,0.5048,0.5030,0.6188,0.5158,-0.0027,-0.0025,-0.0168,-0.0039
3,QQQ,+ technical indicators minus base,Majority baseline,0.5037,0.5000,0.6700,NaN,0.5037,0.5000,0.6700,NaN,0.0000,0.0000,0.0000,NaN
4,QQQ,+ technical indicators minus base,XGBoost,0.5130,0.5120,0.5721,0.5183,0.5154,0.5145,0.5744,0.5245,-0.0025,-0.0025,-0.0023,-0.0062
5,SPY,+ technical indicators minus base,Direction persistence baseline,0.4733,0.4732,0.4705,NaN,0.4733,0.4732,0.4705,NaN,0.0000,0.0000,0.0000,NaN
6,SPY,+ technical indicators minus base,LSTM,0.5139,0.5138,0.5902,0.5283,0.5139,0.5140,0.5973,0.5281,-0.0000,-0.0002,-0.0070,0.0002
7,SPY,+ technical indicators minus base,Logistic regression,0.5133,0.5131,0.5890,0.5242,0.5164,0.5164,0.5929,0.5268,-0.0031,-0.0033,-0.0039,-0.0026
8,SPY,+ technical indicators minus base,Majority baseline,0.4999,0.5000,0.6665,NaN,0.4999,0.5000,0.6665,NaN,0.0000,0.0000,0.0000,NaN
9,SPY,+ technical indicators minus base,XGBoost,0.5144,0.5148,0.5686,0.5266,0.5147,0.5150,0.5718,0.5287,-0.0002,-0.0001,-0.0032,-0.0020


## Cost-sensitive XGBoost backtest

In [3]:
xgb_backtest = backtest.loc[backtest['model'].eq('XGBoost')].copy()
display(xgb_backtest.sort_values(['ETF', 'feature_set', 'cost_bps_per_side']).round(4))

,ETF,model,cost_bps_per_side,intervals,active_intervals,exposure_fraction,entries,exits,trade_events,one_way_turnover,entries_per_session,cumulative_cost_fraction_uncompounded,sessions,gross_cumulative_return,net_cumulative_return,maximum_drawdown,annualized_daily_sharpe,feature_set
15,QQQ,XGBoost,0.0,13598,8652,0.6363,2805,2805,5610,5610.0,6.4483,0.0000,435,0.1695,0.1695,-0.0437,1.3704,+ technical indicators
16,QQQ,XGBoost,1.0,13598,8652,0.6363,2805,2805,5610,5610.0,6.4483,0.5610,435,0.1695,-0.3327,-0.3342,-3.4527,+ technical indicators
17,QQQ,XGBoost,2.5,13598,8652,0.6363,2805,2805,5610,5610.0,6.4483,1.4025,435,0.1695,-0.7124,-0.7124,-10.6310,+ technical indicators
18,QQQ,XGBoost,5.0,13598,8652,0.6363,2805,2805,5610,5610.0,6.4483,2.8050,435,0.1695,-0.9293,-0.9293,-21.2712,+ technical indicators
19,QQQ,XGBoost,10.0,13598,8652,0.6363,2805,2805,5610,5610.0,6.4483,5.6100,435,0.1695,-0.9957,-0.9957,-34.7403,+ technical indicators
40,QQQ,XGBoost,0.0,13598,8657,0.6366,3127,3127,6254,6254.0,7.1885,0.0000,435,0.1700,0.1700,-0.0435,1.6018,Base features
41,QQQ,XGBoost,1.0,13598,8657,0.6366,3127,3127,6254,6254.0,7.1885,0.6254,435,0.1700,-0.3740,-0.3743,-4.6913,Base features
42,QQQ,XGBoost,2.5,13598,8657,0.6366,3127,3127,6254,6254.0,7.1885,1.5635,435,0.1700,-0.7550,-0.7550,-13.9273,Base features
43,QQQ,XGBoost,5.0,13598,8657,0.6366,3127,3127,6254,6254.0,7.1885,3.1270,435,0.1700,-0.9487,-0.9487,-27.1120,Base features
44,QQQ,XGBoost,10.0,13598,8657,0.6366,3127,3127,6254,6254.0,7.1885,6.2540,435,0.1700,-0.9978,-0.9978,-42.6085,Base features


## Coverage and timestamp alignment

The number of out-of-sample predictions and unique timestamps should match across the two feature sets for each ETF.

In [4]:
coverage = (predictions.groupby(['ETF', 'feature_set'])
            .agg(rows=('datetime', 'size'), unique_timestamps=('datetime', 'nunique'))
            .reset_index())
display(coverage)
assert predictions.groupby('ETF')['feature_set'].nunique().eq(2).all()
assert not predictions.duplicated(['ETF', 'feature_set', 'datetime']).any()

,ETF,feature_set,rows,unique_timestamps
0,QQQ,+ technical indicators,13598,13598
1,QQQ,Base features,13598,13598
2,SPY,+ technical indicators,13558,13558
3,SPY,Base features,13558,13558
